# Tomato Leaf Disease Classification Using Deep Learning

**Authors:** Research Group  
**Framework:** PyTorch  
**Dataset:** PlantVillage — Tomato Subset  
**Objective:** Multi-class classification of tomato leaf diseases using CNN and Transfer Learning

---

## Abstract

Tomato crop diseases represent a significant threat to agricultural yield worldwide. Early and accurate identification of foliar diseases is critical for timely intervention and minimizing economic losses. In this work, we develop and evaluate a deep learning pipeline for automated classification of tomato leaf diseases using the PlantVillage dataset. We begin with a custom Convolutional Neural Network (CNN) baseline and progressively improve the architecture through transfer learning with ResNet50 and subsequent fine-tuning. Our evaluation framework includes class-level precision, recall, F1 scores, and confusion matrix analysis, providing a comprehensive view of model behaviour across all disease categories. The trained models are serialized for downstream integration into a Flask-based backend API, enabling real-time inference within a full-stack web application.

---

## Research Pipeline

The following stages constitute the end-to-end research pipeline implemented in this notebook:

1. Environment Setup and Library Imports
2. Dataset Analysis and Exploratory Data Analysis (EDA)
3. Data Preprocessing and Augmentation
4. Custom CNN Baseline Model
5. Transfer Learning with ResNet50
6. Fine-Tuning and Hyperparameter Refinement
7. Comprehensive Model Evaluation
8. Model Serialization for Production Deployment


## Section 1 — Environment Setup and Library Imports

Before initiating any experimentation, we import the full stack of required libraries. PyTorch serves as the deep learning backbone, while scikit-learn provides evaluation utilities. Matplotlib and Seaborn are used for visualization throughout the notebook. We also configure the computation device at this stage, defaulting to GPU acceleration whenever a CUDA-capable device is available — this is critical for reducing training time on image datasets of this scale.


In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split, Subset
from torchvision import transforms, datasets, models

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

warnings.filterwarnings("ignore")

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ── Device Configuration ─────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Computation device : {device}")
if device.type == "cuda":
    print(f"GPU                : {torch.cuda.get_device_name(0)}")
    print(f"CUDA version       : {torch.version.cuda}")
else:
    print("Running on CPU — consider enabling GPU for faster training.")

# ── Plotting Style ────────────────────────────────────────────────────────────
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 11
sns.set_style("whitegrid")

print("\nAll libraries loaded successfully.")

## Section 2 — Global Configuration

Centralizing all hyperparameters and path configurations into a single dictionary makes experiments reproducible and straightforward to modify. Rather than scattering magic numbers throughout the notebook, every tunable value is defined here and referenced by name downstream. This practice is standard in research-grade codebases.


In [ ]:
# ── Global Configuration ─────────────────────────────────────────────────────
CONFIG = {
    # Paths
    "data_dir"       : "/kaggle/input/tomato-leaf-disease",
    "model_save_dir" : "/kaggle/working",

    # Input
    "img_size"       : 224,
    "num_workers"    : 2,

    # Training
    "batch_size"     : 32,
    "seed"           : 42,

    # Baseline CNN
    "cnn_epochs"     : 10,
    "cnn_lr"         : 1e-3,

    # Transfer Learning — frozen head only
    "tl_epochs"      : 5,
    "tl_lr"          : 1e-3,

    # Fine-Tuning — full network
    "ft_epochs"      : 5,
    "ft_lr"          : 1e-5,

    # ImageNet normalization statistics
    "mean"           : [0.485, 0.456, 0.406],
    "std"            : [0.229, 0.224, 0.225],

    # Train / validation split ratio
    "train_split"    : 0.80,
}

print("Configuration summary:")
for k, v in CONFIG.items():
    print(f"  {k:<20} : {v}")

## Section 3 — Dataset Discovery

We begin by inspecting the raw directory structure of the PlantVillage tomato subset. This step confirms that the dataset is organized in the standard ImageFolder format — one subdirectory per class — and allows us to count the total number of samples before committing to any particular preprocessing strategy.


In [ ]:
data_dir = Path(CONFIG["data_dir"])

# ── Directory Traversal ───────────────────────────────────────────────────────
class_counts = {}

for class_dir in sorted(data_dir.iterdir()):
    if class_dir.is_dir():
        n_images = len(list(class_dir.glob("*.jpg")) + list(class_dir.glob("*.JPG")))
        class_counts[class_dir.name] = n_images

total_images = sum(class_counts.values())
num_classes  = len(class_counts)

print(f"Dataset root    : {data_dir}")
print(f"Total classes   : {num_classes}")
print(f"Total images    : {total_images:,}")
print()
print(f"{'Class':<45} {'Count':>6}  {'% of Total':>10}")
print("-" * 65)

for cls, count in sorted(class_counts.items(), key=lambda x: -x[1]):
    print(f"{cls:<45} {count:>6}  {100*count/total_images:>9.1f}%")

## Section 4 — Exploratory Data Analysis

### 4.1 Class Distribution

Understanding the distribution of samples across classes is one of the most important preliminary steps in any classification task. A severely imbalanced dataset can bias the model toward majority classes, leading to deceptively high accuracy while minority classes are effectively ignored. The bar chart below provides a clear visual summary of per-class sample counts.


In [ ]:
df_counts = (
    pd.DataFrame(list(class_counts.items()), columns=["Class", "Count"])
    .sort_values("Count", ascending=False)
    .reset_index(drop=True)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart
axes[0].bar(df_counts["Class"], df_counts["Count"], color=sns.color_palette("viridis", num_classes))
axes[0].set_xticklabels(df_counts["Class"], rotation=90, fontsize=8)
axes[0].set_title("Per-Class Image Counts", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Number of Images")
axes[0].set_xlabel("Disease Class")

# Pie chart — top 5 vs rest
top5     = df_counts.head(5)
rest_sum = df_counts.iloc[5:]["Count"].sum()
pie_data = list(top5["Count"]) + [rest_sum]
pie_lbls = list(top5["Class"]) + [f"Others ({num_classes-5} classes)"]

axes[1].pie(pie_data, labels=pie_lbls, autopct="%1.1f%%", startangle=140,
            textprops={"fontsize": 8})
axes[1].set_title("Top-5 Classes vs. Others", fontsize=13, fontweight="bold")

plt.suptitle("PlantVillage — Tomato Subset: Class Distribution", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\nDescriptive statistics:")
print(df_counts["Count"].describe().to_string())

### 4.2 Sample Image Visualization

Before training any model, it is good practice to visually inspect representative images from each class. This helps identify potential data quality issues, class overlap, and the level of visual complexity the model will need to handle. We display one sample image from each class, annotated with its class label.


In [ ]:
classes_list = sorted(class_counts.keys())
n_cols       = 5
n_rows       = int(np.ceil(len(classes_list) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
axes      = axes.flatten()

for idx, class_name in enumerate(classes_list):
    class_path = data_dir / class_name
    img_files  = list(class_path.glob("*.jpg")) + list(class_path.glob("*.JPG"))
    if not img_files:
        continue
    img = Image.open(img_files[0]).convert("RGB")
    axes[idx].imshow(img)
    axes[idx].set_title(class_name.replace("Tomato_", "").replace("_", " "),
                        fontsize=8, wrap=True)
    axes[idx].axis("off")

# Hide unused axes
for ax in axes[len(classes_list):]:
    ax.set_visible(False)

plt.suptitle("Representative Samples — One Image per Disease Class",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 4.3 Image Dimension Analysis

Knowing the native resolution of the images informs our choice of resize target. If most images are already close to 224×224 pixels, downscaling is lossless. If they are significantly larger, we risk losing fine-grained texture information relevant to disease identification.


In [ ]:
widths, heights = [], []

# Sample up to 50 images per class to keep this cell fast
for class_name in classes_list:
    class_path = data_dir / class_name
    img_files  = list(class_path.glob("*.jpg")) + list(class_path.glob("*.JPG"))
    for f in img_files[:50]:
        w, h = Image.open(f).size
        widths.append(w)
        heights.append(h)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(widths, bins=30, color="steelblue", edgecolor="white")
axes[0].set_title("Width Distribution", fontweight="bold")
axes[0].set_xlabel("Pixels")
axes[0].set_ylabel("Frequency")
axes[0].axvline(np.mean(widths), color="red", linestyle="--", label=f"Mean = {np.mean(widths):.0f}px")
axes[0].legend()

axes[1].hist(heights, bins=30, color="coral", edgecolor="white")
axes[1].set_title("Height Distribution", fontweight="bold")
axes[1].set_xlabel("Pixels")
axes[1].set_ylabel("Frequency")
axes[1].axvline(np.mean(heights), color="red", linestyle="--", label=f"Mean = {np.mean(heights):.0f}px")
axes[1].legend()

plt.suptitle("Native Image Dimension Distribution (Sampled)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"Mean width   : {np.mean(widths):.0f} px  |  Std: {np.std(widths):.0f} px")
print(f"Mean height  : {np.mean(heights):.0f} px  |  Std: {np.std(heights):.0f} px")
print(f"Min size     : {min(widths)}×{min(heights)}")
print(f"Max size     : {max(widths)}×{max(heights)}")
print(f"\nAll images will be resized to {CONFIG['img_size']}×{CONFIG['img_size']} during preprocessing.")

## Section 5 — Data Preprocessing and Augmentation

### 5.1 Transform Pipelines

Data augmentation is a widely used regularization technique in computer vision that artificially increases the effective size and diversity of the training set by applying random geometric and photometric transformations. We apply augmentation exclusively during training — the validation set receives only deterministic resizing and normalization to ensure unbiased evaluation.

The normalization statistics (mean and standard deviation computed from ImageNet) are standard when using pretrained torchvision models, as the convolutional filters of those models were trained under these exact input statistics.


In [ ]:
img_size = CONFIG["img_size"]
mean     = CONFIG["mean"]
std      = CONFIG["std"]

# ── Training Transforms ───────────────────────────────────────────────────────
train_transforms = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

# ── Validation / Inference Transforms ────────────────────────────────────────
val_transforms = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

print("Training transform pipeline:")
print(train_transforms)
print()
print("Validation transform pipeline:")
print(val_transforms)

### 5.2 Dataset Loading and Train-Validation Split

We use PyTorch's `ImageFolder` utility to automatically infer class labels from subdirectory names. The dataset is then partitioned into 80% training and 20% validation using a reproducible random split. An important implementation note: the validation subset must have the deterministic transform applied, not the augmentation pipeline — this is achieved by reassigning `transform` on the validation `Subset`.


In [ ]:
# ── Load with Training Transforms Initially ───────────────────────────────────
full_dataset = datasets.ImageFolder(CONFIG["data_dir"], transform=train_transforms)

# ── Class Metadata ────────────────────────────────────────────────────────────
class_names = full_dataset.classes
class_to_idx = full_dataset.class_to_idx
num_classes  = len(class_names)

# ── Reproducible Split ─────────────────────────────────────────────────────────
train_size = int(CONFIG["train_split"] * len(full_dataset))
val_size   = len(full_dataset) - train_size

generator  = torch.Generator().manual_seed(CONFIG["seed"])
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size],
                                          generator=generator)

# ── Apply Correct Transforms to Validation Set ────────────────────────────────
# We need a separate ImageFolder instance with val_transforms for the val subset.
val_dataset_full = datasets.ImageFolder(CONFIG["data_dir"], transform=val_transforms)
val_dataset      = Subset(val_dataset_full, val_dataset.indices)

# ── DataLoaders ──────────────────────────────────────────────────────────────
train_loader = DataLoader(
    train_dataset,
    batch_size  = CONFIG["batch_size"],
    shuffle     = True,
    num_workers = CONFIG["num_workers"],
    pin_memory  = (device.type == "cuda"),
)

val_loader = DataLoader(
    val_dataset,
    batch_size  = CONFIG["batch_size"],
    shuffle     = False,
    num_workers = CONFIG["num_workers"],
    pin_memory  = (device.type == "cuda"),
)

print(f"Number of classes    : {num_classes}")
print(f"Total samples        : {len(full_dataset):,}")
print(f"Training samples     : {len(train_dataset):,}")
print(f"Validation samples   : {len(val_dataset):,}")
print(f"Training batches     : {len(train_loader)}")
print(f"Validation batches   : {len(val_loader)}")
print()
print("Class-to-index mapping (first 5):")
for cls, idx in list(class_to_idx.items())[:5]:
    print(f"  {idx:>2}  {cls}")

### 5.3 Visualizing Augmented Samples

To confirm that augmentation is applied correctly, we display several augmented versions of the same source image side-by-side. Each instance corresponds to a different random draw from the stochastic transform pipeline, demonstrating the range of transformations the model will encounter during training.


In [ ]:
# ── Inverse normalization for visualization ───────────────────────────────────
inv_mean = [-m / s for m, s in zip(CONFIG["mean"], CONFIG["std"])]
inv_std  = [1.0 / s for s in CONFIG["std"]]
inv_norm = transforms.Normalize(mean=inv_mean, std=inv_std)

# Load one raw image
sample_class = class_names[0]
sample_path  = list((data_dir / sample_class).glob("*.jpg"))[0]
raw_img      = Image.open(sample_path).convert("RGB")

# Generate augmented versions
n_aug  = 8
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes   = axes.flatten()

# Original
axes[0].imshow(raw_img.resize((224, 224)))
axes[0].set_title("Original", fontweight="bold")
axes[0].axis("off")

for i in range(1, n_aug):
    aug_tensor = train_transforms(raw_img)
    aug_vis    = inv_norm(aug_tensor).permute(1, 2, 0).clamp(0, 1).numpy()
    axes[i].imshow(aug_vis)
    axes[i].set_title(f"Augmented {i}")
    axes[i].axis("off")

plt.suptitle(f"Data Augmentation Samples  —  Class: {sample_class}",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Section 6 — Training and Evaluation Utilities

### 6.1 Core Training Loop

We encapsulate training logic in a reusable function that accepts any model, optimizer, and number of epochs. This avoids code duplication across baseline, transfer learning, and fine-tuning experiments. The function returns per-epoch loss and accuracy arrays to facilitate learning curve visualization.


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    """Run one complete pass over the training data and return loss + accuracy."""
    model.train()
    running_loss    = 0.0
    correct         = 0
    total           = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted  = torch.max(outputs, 1)
        correct      += (predicted == labels).sum().item()
        total        += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc  = 100.0 * correct / total
    return epoch_loss, epoch_acc


def evaluate(model, loader, criterion):
    """Evaluate model on loader — returns loss and accuracy."""
    model.eval()
    running_loss = 0.0
    correct      = 0
    total        = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs         = model(images)
            loss            = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted  = torch.max(outputs, 1)
            correct      += (predicted == labels).sum().item()
            total        += labels.size(0)

    return running_loss / total, 100.0 * correct / total


def run_training(model, train_loader, val_loader, criterion, optimizer,
                 n_epochs, scheduler=None, experiment_name="Experiment"):
    """
    Full training loop with per-epoch reporting and history tracking.

    Returns
    -------
    history : dict  —  keys: train_loss, val_loss, train_acc, val_acc
    """
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    print(f"\n{'═'*65}")
    print(f"  {experiment_name}")
    print(f"{'═'*65}")
    print(f"{'Epoch':<8} {'Train Loss':>10} {'Train Acc':>10} {'Val Loss':>10} {'Val Acc':>10}")
    print(f"{'─'*50}")

    for epoch in range(1, n_epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        vl_loss, vl_acc = evaluate(model, val_loader, criterion)

        if scheduler is not None:
            scheduler.step()

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(vl_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(vl_acc)

        print(f"{epoch:<8} {tr_loss:>10.4f} {tr_acc:>9.2f}% {vl_loss:>10.4f} {vl_acc:>9.2f}%")

    print(f"{'─'*50}")
    print(f"Best Val Accuracy : {max(history['val_acc']):.2f}%")
    return history


def plot_history(history, title="Learning Curves"):
    """Plot loss and accuracy curves for a single training run."""
    epochs = range(1, len(history["train_loss"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(epochs, history["train_loss"], "b-o", label="Train Loss",  linewidth=2)
    axes[0].plot(epochs, history["val_loss"],   "r-o", label="Val Loss",    linewidth=2)
    axes[0].set_title(f"{title} — Loss",  fontweight="bold")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Cross-Entropy Loss")
    axes[0].legend()

    axes[1].plot(epochs, history["train_acc"], "b-o", label="Train Acc",  linewidth=2)
    axes[1].plot(epochs, history["val_acc"],   "r-o", label="Val Acc",    linewidth=2)
    axes[1].set_title(f"{title} — Accuracy", fontweight="bold")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy (%)")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


print("Training utilities defined successfully.")

## Section 7 — Baseline Custom CNN

### 7.1 Architecture Definition

We construct a straightforward three-block convolutional neural network as our baseline. Each convolutional block follows the standard pattern of convolution → batch normalization → ReLU activation → max pooling. Batch normalization is included to stabilize training and reduce sensitivity to initialization. A single fully connected head with dropout regularization maps the flattened feature map to class logits.

This baseline serves two purposes: it establishes a minimum performance threshold for transfer learning to improve upon, and it gives us insight into the raw difficulty of the classification task without ImageNet priors.


In [ ]:
class TomatoCNN(nn.Module):
    """
    Three-block custom CNN for tomato leaf disease classification.

    Architecture
    ------------
    Block 1 : Conv(3→32)   → BN → ReLU → MaxPool
    Block 2 : Conv(32→64)  → BN → ReLU → MaxPool
    Block 3 : Conv(64→128) → BN → ReLU → MaxPool
    Head    : Flatten → FC(512) → ReLU → Dropout(0.5) → FC(num_classes)
    """

    def __init__(self, num_classes: int):
        super(TomatoCNN, self).__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                      # 224 → 112

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                      # 112 → 56

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                      # 56  → 28
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.50),
            nn.Linear(512, num_classes),
        )

        self._initialize_weights()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.classifier(x)
        return x

    def _initialize_weights(self):
        """Kaiming Normal initialization for convolutional layers."""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.constant_(m.bias, 0)


# ── Instantiate and Inspect ───────────────────────────────────────────────────
cnn_model = TomatoCNN(num_classes=num_classes).to(device)

# Parameter count
total_params     = sum(p.numel() for p in cnn_model.parameters())
trainable_params = sum(p.numel() for p in cnn_model.parameters() if p.requires_grad)

print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")

### 7.2 Baseline CNN Training

We use the Adam optimizer with a cosine annealing learning rate schedule, which smoothly reduces the learning rate from the initial value to near zero over the course of training. This encourages the model to converge to a flatter, more generalizable minimum compared to a fixed learning rate.


In [ ]:
criterion_cnn = nn.CrossEntropyLoss()

optimizer_cnn = optim.Adam(
    cnn_model.parameters(),
    lr           = CONFIG["cnn_lr"],
    weight_decay = 1e-4,
)

scheduler_cnn = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_cnn,
    T_max = CONFIG["cnn_epochs"],
)

history_cnn = run_training(
    model           = cnn_model,
    train_loader    = train_loader,
    val_loader      = val_loader,
    criterion       = criterion_cnn,
    optimizer       = optimizer_cnn,
    n_epochs        = CONFIG["cnn_epochs"],
    scheduler       = scheduler_cnn,
    experiment_name = "Baseline CNN — TomatoCNN",
)

plot_history(history_cnn, title="Baseline CNN")

## Section 8 — Transfer Learning with ResNet50

### 8.1 Loading and Adapting the Pretrained Network

ResNet50 pre-trained on ImageNet provides a powerful feature extractor whose lower-layer filters detect general visual primitives (edges, textures, colour blobs) that are equally useful for plant disease recognition. We replace only the final fully connected layer to match our target number of classes, while freezing all other parameters. This stage is sometimes called "linear probing" and is used to warm-start the classification head before full fine-tuning.


In [ ]:
# ── Load Pretrained ResNet50 ──────────────────────────────────────────────────
resnet50 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

# Freeze all parameters
for param in resnet50.parameters():
    param.requires_grad = False

# Replace the classification head
in_features   = resnet50.fc.in_features
resnet50.fc   = nn.Sequential(
    nn.Linear(in_features, 512),
    nn.ReLU(inplace=True),
    nn.Dropout(p=0.40),
    nn.Linear(512, num_classes),
)

resnet50 = resnet50.to(device)

# ── Parameter Summary ─────────────────────────────────────────────────────────
total_p     = sum(p.numel() for p in resnet50.parameters())
trainable_p = sum(p.numel() for p in resnet50.parameters() if p.requires_grad)
frozen_p    = total_p - trainable_p

print(f"Total parameters     : {total_p:,}")
print(f"Trainable parameters : {trainable_p:,}  (head only — frozen backbone)")
print(f"Frozen parameters    : {frozen_p:,}")

### 8.2 Training the Classification Head

We train only the newly added classification head using a relatively high learning rate. Since the backbone weights are frozen, this phase runs quickly and serves to bring the head weights to a sensible initialization before fine-tuning the entire network.


In [ ]:
criterion_tl = nn.CrossEntropyLoss()

optimizer_tl = optim.Adam(
    filter(lambda p: p.requires_grad, resnet50.parameters()),
    lr = CONFIG["tl_lr"],
)

history_tl = run_training(
    model           = resnet50,
    train_loader    = train_loader,
    val_loader      = val_loader,
    criterion       = criterion_tl,
    optimizer       = optimizer_tl,
    n_epochs        = CONFIG["tl_epochs"],
    experiment_name = "Transfer Learning — ResNet50 (Frozen Backbone)",
)

plot_history(history_tl, title="Transfer Learning — Frozen Backbone")

## Section 9 — Fine-Tuning the Full Network

### 9.1 Unfreezing the Backbone

After the classification head has converged to a reasonable solution, we unfreeze all layers and continue training the entire network with a substantially smaller learning rate. Fine-tuning allows the pretrained feature representations to adapt to the specific visual patterns of tomato diseases — subtle colour variations, lesion shapes, and surface textures — that may differ from the broader distribution of ImageNet images.

Using a learning rate that is one to two orders of magnitude smaller than the initial head-training phase prevents destructive interference with the pretrained weights during the early iterations of fine-tuning.


In [ ]:
# Unfreeze all layers
for param in resnet50.parameters():
    param.requires_grad = True

trainable_p = sum(p.numel() for p in resnet50.parameters() if p.requires_grad)
print(f"Trainable parameters after unfreeze : {trainable_p:,}")

optimizer_ft = optim.Adam(
    resnet50.parameters(),
    lr           = CONFIG["ft_lr"],
    weight_decay = 1e-5,
)

scheduler_ft = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_ft,
    T_max = CONFIG["ft_epochs"],
)

history_ft = run_training(
    model           = resnet50,
    train_loader    = train_loader,
    val_loader      = val_loader,
    criterion       = criterion_tl,
    optimizer       = optimizer_ft,
    n_epochs        = CONFIG["ft_epochs"],
    scheduler       = scheduler_ft,
    experiment_name = "Fine-Tuning — ResNet50 (Full Network)",
)

plot_history(history_ft, title="Fine-Tuning — Full ResNet50")

## Section 10 — Comprehensive Model Evaluation

### 10.1 Generating Predictions

We collect all predictions on the held-out validation set in a single inference pass. Disabling gradient computation via `torch.no_grad()` reduces memory overhead and accelerates the forward pass.


In [ ]:
def get_all_predictions(model, loader):
    """
    Run inference over an entire DataLoader.

    Returns
    -------
    preds  : np.ndarray  —  predicted class indices
    labels : np.ndarray  —  ground-truth class indices
    probs  : np.ndarray  —  softmax probabilities (shape: N × num_classes)
    """
    model.eval()
    all_preds  = []
    all_labels = []
    all_probs  = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            logits = model(images)
            probs  = F.softmax(logits, dim=1)
            _, predicted = torch.max(logits, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())

    return np.array(all_preds), np.array(all_labels), np.array(all_probs)


preds, labels, probs = get_all_predictions(resnet50, val_loader)

overall_acc = accuracy_score(labels, preds) * 100
overall_f1  = f1_score(labels, preds, average="weighted") * 100

print(f"Overall Validation Accuracy : {overall_acc:.2f}%")
print(f"Weighted F1 Score           : {overall_f1:.2f}%")

### 10.2 Per-Class Classification Report

The classification report decomposes model performance by class, reporting precision, recall, and F1 score for each disease category. This is substantially more informative than aggregate accuracy alone, particularly for imbalanced datasets where some classes may have systematically lower recall.


In [ ]:
report = classification_report(labels, preds, target_names=class_names, digits=3)
print("Classification Report")
print("=" * 80)
print(report)

# ── Per-class F1 for plotting ─────────────────────────────────────────────────
from sklearn.metrics import classification_report as cr
report_dict = cr(labels, preds, target_names=class_names, output_dict=True)
per_class_f1 = {cls: report_dict[cls]["f1-score"] for cls in class_names}

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(class_names, [v*100 for v in per_class_f1.values()],
              color=sns.color_palette("RdYlGn", num_classes))
ax.set_xticklabels(class_names, rotation=90, fontsize=8)
ax.set_ylim(0, 110)
ax.set_ylabel("F1 Score (%)")
ax.set_title("Per-Class F1 Score — Fine-Tuned ResNet50", fontweight="bold")
ax.axhline(overall_f1, color="navy", linestyle="--", label=f"Weighted Avg F1 = {overall_f1:.1f}%")
ax.legend()
plt.tight_layout()
plt.show()

### 10.3 Confusion Matrix

The confusion matrix provides a detailed breakdown of correct and incorrect predictions, revealing systematic misclassification patterns. Off-diagonal entries that are concentrated in specific class pairs indicate visually similar diseases that the model struggles to distinguish — this is actionable information for targeted data collection or architecture improvements.


In [ ]:
cm = confusion_matrix(labels, preds)

# Normalized confusion matrix (row-normalized → recall per class)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

sns.heatmap(cm, ax=axes[0], cmap="Blues",
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.3, linecolor="white",
            cbar_kws={"label": "Count"})
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=90, fontsize=7)
axes[0].set_yticklabels(axes[0].get_yticklabels(), rotation=0,  fontsize=7)
axes[0].set_title("Confusion Matrix (Counts)", fontweight="bold")
axes[0].set_xlabel("Predicted Class")
axes[0].set_ylabel("True Class")

sns.heatmap(cm_norm, ax=axes[1], cmap="YlOrRd",
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.3, linecolor="white", vmin=0, vmax=1,
            cbar_kws={"label": "Recall (Row-Normalized)"})
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=90, fontsize=7)
axes[1].set_yticklabels(axes[1].get_yticklabels(), rotation=0,  fontsize=7)
axes[1].set_title("Normalized Confusion Matrix (Recall)", fontweight="bold")
axes[1].set_xlabel("Predicted Class")
axes[1].set_ylabel("True Class")

plt.suptitle("Validation Set Confusion Matrices — Fine-Tuned ResNet50",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

### 10.4 Experiment Comparison

We aggregate the validation accuracy achieved at the end of each training phase to directly compare the impact of the different modelling strategies.


In [ ]:
comparison = {
    "Baseline CNN"              : max(history_cnn["val_acc"]),
    "Transfer Learning (Frozen)": max(history_tl["val_acc"]),
    "Fine-Tuned ResNet50"       : max(history_ft["val_acc"]),
}

print("\n" + "="*45)
print(f"{'Experiment':<30} {'Val Acc':>10}")
print("="*45)
for exp, acc in comparison.items():
    print(f"{exp:<30} {acc:>9.2f}%")
print("="*45)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ["#4e79a7", "#f28e2b", "#59a14f"]
bars   = ax.bar(comparison.keys(), comparison.values(), color=colors, edgecolor="white", linewidth=1.2)
ax.set_ylim(0, 110)
ax.set_ylabel("Validation Accuracy (%)", fontsize=12)
ax.set_title("Experiment Comparison — Validation Accuracy", fontsize=13, fontweight="bold")
for bar, val in zip(bars, comparison.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
            f"{val:.2f}%", ha="center", va="bottom", fontweight="bold")
plt.tight_layout()
plt.show()

### 10.5 Qualitative Sample Predictions

Qualitative inspection of model predictions on validation images provides intuitive confirmation that the model is responding to the right visual cues rather than spurious correlations. We display both correctly classified and misclassified examples to better understand failure modes.


In [ ]:
def show_predictions(model, loader, class_names, n=16):
    """Display a grid of validation images with true and predicted labels."""
    model.eval()
    images_shown = []
    preds_shown  = []
    labels_shown = []

    with torch.no_grad():
        for imgs, lbls in loader:
            logits = model(imgs.to(device))
            _, pred = torch.max(logits, 1)
            for i in range(len(imgs)):
                if len(images_shown) >= n:
                    break
                images_shown.append(imgs[i])
                preds_shown.append(pred[i].item())
                labels_shown.append(lbls[i].item())
            if len(images_shown) >= n:
                break

    n_cols = 4
    n_rows = int(np.ceil(n / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
    axes = axes.flatten()

    inv_mean = [-m/s for m,s in zip(CONFIG["mean"], CONFIG["std"])]
    inv_std  = [1.0/s for s in CONFIG["std"]]
    inv_norm = transforms.Normalize(mean=inv_mean, std=inv_std)

    for i in range(n):
        img_vis = inv_norm(images_shown[i]).permute(1,2,0).clamp(0,1).numpy()
        axes[i].imshow(img_vis)

        pred_cls  = class_names[preds_shown[i]].replace("Tomato_","").replace("_"," ")
        true_cls  = class_names[labels_shown[i]].replace("Tomato_","").replace("_"," ")
        correct   = preds_shown[i] == labels_shown[i]

        axes[i].set_title(
            f"True : {true_cls}\nPred : {pred_cls}",
            color="green" if correct else "red",
            fontsize=7,
        )
        axes[i].axis("off")

    for ax in axes[n:]:
        ax.set_visible(False)

    plt.suptitle("Sample Predictions  |  Green = Correct  |  Red = Incorrect",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()


show_predictions(resnet50, val_loader, class_names, n=16)

## Section 11 — Model Serialization for Deployment

### 11.1 Saving Model Weights

We persist the fine-tuned ResNet50 state dictionary for subsequent use in the Flask-based REST API. Two artefacts are saved: the raw PyTorch `.pth` weight file and a JSON metadata file containing the class-to-index mapping. The metadata file is required by the inference endpoint to correctly decode predicted indices back to human-readable class names.


In [ ]:
import json
from pathlib import Path

save_dir = Path(CONFIG["model_save_dir"])
save_dir.mkdir(parents=True, exist_ok=True)

# ── Save Model Weights ────────────────────────────────────────────────────────
model_path = save_dir / "tomato_disease_resnet50.pth"
torch.save(resnet50.state_dict(), model_path)
print(f"Model weights saved to : {model_path}")

# ── Save Class Metadata ───────────────────────────────────────────────────────
metadata = {
    "class_names"  : class_names,
    "class_to_idx" : class_to_idx,
    "idx_to_class" : {str(v): k for k, v in class_to_idx.items()},
    "num_classes"  : num_classes,
    "img_size"     : CONFIG["img_size"],
    "mean"         : CONFIG["mean"],
    "std"          : CONFIG["std"],
}

metadata_path = save_dir / "class_metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Class metadata saved to: {metadata_path}")
print()
print("Saved class names:")
for i, cls in enumerate(class_names):
    print(f"  {i:>2}  {cls}")

### 11.2 Standalone Inference Function

The following function mirrors exactly what the Flask API will execute at runtime. It accepts either a raw file path or a PIL Image, applies the deterministic preprocessing pipeline, and returns the top-K predictions with confidence scores. Defining this function in the notebook allows us to validate end-to-end inference before wiring it into the web application.


In [ ]:
def predict_disease(image_input, model, class_names, top_k=5):
    """
    Run inference on a single image and return top-K predictions.

    Parameters
    ----------
    image_input : str | Path | PIL.Image
        Input image. Accepts file path or PIL Image object.
    model : nn.Module
        Loaded PyTorch model in eval mode.
    class_names : list[str]
        Ordered list of class names aligned with model output indices.
    top_k : int
        Number of top predictions to return.

    Returns
    -------
    results : list[dict]
        Each entry contains 'class', 'confidence' (0–100%), and 'index'.
    """
    if isinstance(image_input, (str, Path)):
        img = Image.open(image_input).convert("RGB")
    else:
        img = image_input.convert("RGB")

    transform = transforms.Compose([
        transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
        transforms.ToTensor(),
        transforms.Normalize(mean=CONFIG["mean"], std=CONFIG["std"]),
    ])

    tensor = transform(img).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits      = model(tensor)
        probs       = F.softmax(logits, dim=1).squeeze()
        top_k_probs, top_k_indices = torch.topk(probs, k=min(top_k, len(class_names)))

    results = [
        {
            "rank"       : i + 1,
            "class"      : class_names[idx.item()],
            "confidence" : round(prob.item() * 100, 2),
            "index"      : idx.item(),
        }
        for i, (prob, idx) in enumerate(zip(top_k_probs, top_k_indices))
    ]

    return results


# ── Demo Inference ────────────────────────────────────────────────────────────
sample_img_path = list((data_dir / class_names[0]).glob("*.jpg"))[0]
results = predict_disease(sample_img_path, resnet50, class_names, top_k=5)

print(f"Input image : {sample_img_path.name}")
print(f"\n{'Rank':<6} {'Class':<45} {'Confidence':>12}")
print("-" * 65)
for r in results:
    print(f"{r['rank']:<6} {r['class']:<45} {r['confidence']:>11.2f}%")

## Section 12 — Summary and Next Steps

### 12.1 Research Summary

In this notebook we implemented and evaluated a complete deep learning pipeline for automated tomato leaf disease classification. The key findings are summarized below.

**Dataset:** The PlantVillage tomato subset contains images spanning multiple disease categories. Class distribution shows some imbalance that should be monitored if classification performance on minority classes is a priority in production.

**Baseline CNN:** The custom three-block CNN provides a reasonable baseline. Its relatively modest parameter count keeps inference latency low but limits representational capacity for this visually complex task.

**Transfer Learning:** Fine-tuning a ResNet50 backbone pretrained on ImageNet yields a substantial accuracy improvement. The pretrained convolutional features provide a rich representation space that generalizes well to disease-related visual patterns.

**Evaluation:** Per-class F1 scores reveal that visually similar disease categories (for example, different bacterial spot variants) represent the primary source of confusion. Targeted data collection for these classes would be the most efficient path to further accuracy gains.

### 12.2 Next Steps — Flask + React Integration

The serialized model artefacts (`tomato_disease_resnet50.pth` and `class_metadata.json`) are ready for integration into the full-stack application. The recommended backend workflow is:

1. Load model and metadata once at application startup
2. Accept image uploads via a `/predict` POST endpoint
3. Run `predict_disease()` on each uploaded image
4. Return the JSON prediction response to the React frontend

### 12.3 Potential Improvements

Further research directions include exploring lighter architectures such as EfficientNet-B0 or MobileNetV3 for reduced inference latency, applying test-time augmentation (TTA) to improve prediction stability, and investigating class-weighted loss functions to address any remaining class imbalance effects.
